# Accessing Google Satellite Embeddings in the MAAP ADE

Authors: Harshini Girish(UAH), Rajat Shinde(UAH), Alex Mandel(Development Seed)

Date:October 1, 2025

Description: This notebook demonstrates an end-to-end workflow to discover, access, and export Google’s 64-dimensional Satellite Embeddings (AlphaEarth Foundations) from Earth Engine (EE) into the MAAP ADE environment using the MAAP DPS subtitle-export algorithm. You’ll authenticate to EE, stage credentials in your MAAP S3 workspace, submit a distributed job, monitor it, and then browse the produced assets and run quick sanity checks (tile counts, band names).

## Run This Notebook
To access and run this tutorial within MAAP's Algorithm Development Environment (ADE), please refer to the ["Getting started with the MAAP"](https://docs.maap-project.org/en/latest/getting_started/getting_started.html) section of our documentation.

Disclaimer: it is highly recommended to run a tutorial within MAAP's ADE, which already includes packages specific to MAAP, such as maap-py. Running the tutorial outside of the MAAP ADE may lead to errors.

## Additional Resources
**Earth Engine scripts referenced**
These assets were created and exported **by tile via a loop in Google Earth Engine** here:  
<https://code.earthengine.google.com/?scriptPath=users%2Fmmacander%2Fveg_structure%3Asentinel_1%2Fseasonal_s1_tile_export_v2>  
**Old version:**  
<https://code.earthengine.google.com/?scriptPath=users%2Fpaulmontesano%2Fdefault%3Aseasonal_s1_tile_export>

They can be **viewed here:**  
<https://code.earthengine.google.com/?scriptPath=users%2Fpaulmontesano%2Fdefault%3Aseasonal_s1_view>

Here, the **composites are created and shown on-the-fly:**  
<https://code.earthengine.google.com/?scriptPath=users%2Fpaulmontesano%2Fdefault%3Aseasonal_s1_comps>









## About the Dataset
The Google Satellite Embedding V1 (Annual) dataset is a global, analysis-ready ImageCollection in Earth Engine that provides annual images from 2017 onward, where each 10-meter pixel encodes a 64-dimensional embedding vector summarizing the year’s multi-sensor surface conditions at and around that location. These embeddings compress rich time-series information into compact features suitable for downstream mapping, classification, and change-analysis workflows directly inside Earth Engine.

Source:[GOOGLE_SATELLITE_EMBEDDING_V1_ANNUAL](https://developers.google.com/earth-engine/datasets/catalog/GOOGLE_SATELLITE_EMBEDDING_V1_ANNUAL)

## Install/Import Packages
Make sure the following libraries are installed before running the notebook

In [2]:
import os, sys, glob, datetime, shutil, time
import numpy as np
import pandas as pd
import geopandas as gpd
import shapely
from shapely.geometry import Polygon, box
import matplotlib
import matplotlib.pyplot as plt
import rasterio as rio
import ee

## Environment Setup & User Parameters

In [3]:
sys.path.append('/projects/code/icesat2_boreal/lib')

from maap.maap import MAAP

USER         = "harshinigirish"
MAAP_VERSION = "EXPORT_GEE_v4"
QUEUE        = "maap-dps-worker-8gb"
ALGO_ID      = "do_gee_download_by_subtile"    
IDENTIFIER   = "SAT_EMB_2024_AOI1"

# Dataset constants
ASSET_PATH   = "GOOGLE/SATELLITE_EMBEDDING/V1/ANNUAL"
YEAR         = 2024


## Authenticate to Earth Engine

Initialize Earth Engine; if it raises, run `ee.Authenticate()` once and re-initialize.Do this a single time per kernel/session.

In [4]:
try:
    ee.Initialize()
except Exception:
    ee.Authenticate()
    ee.Initialize()

print("EE authenticated:", True)


EE authenticated: True


## Stage EE Credentials to MAAP S3

Copy `~/.config/earthengine/credentials` to `s3://maap-ops-workspace/<USER>/credentials`.The DPS algorithm reads this path to access EE on your behalf.Without this staged file, the export job will fail because DPS cannot perform authenticated EE operations.


In [5]:
# Local EE credential path
local_creds = os.path.expanduser("~/.config/earthengine/credentials")
assert os.path.exists(local_creds), "Run ee.Authenticate() first to create credentials."

# S3 staging path used by MAAP DPS template
CREDS_FN = f"s3://maap-ops-workspace/{USER}/credentials"
print("Staging →", CREDS_FN)


os.system(f"aws s3 cp {local_creds} {CREDS_FN}")


Staging → s3://maap-ops-workspace/harshinigirish/credentials
upload: .config/earthengine/credentials to s3://maap-ops-workspace/harshinigirish/credentials


0

## Submit MAAP DPS Job

Call `maap.submitJob(...)` with algo id/version, queue, creds path, and `asset_path`. This captures and log the returned `job_id` and initial submission status. This creates a server-side run that will write results to your designated S3 `out_dir`.


In [8]:
maap = MAAP(maap_host='api.maap-project.org')

submitted_job = maap.submitJob(
    identifier = IDENTIFIER,
    algo_id    = ALGO_ID,
    version    = MAAP_VERSION,
    username   = USER,
    queue      = QUEUE,
    creds_fn   = CREDS_FN,
    
    asset_path = ASSET_PATH,
   
    out_dir    = f"s3://maap-ops-workspace/{USER}/gee_exports/sat_emb_{YEAR}/"
)

print("Submit status:", submitted_job.status)   # expect 'success'
job_id = submitted_job.id
print("Job ID:", job_id)


Submit status: success
Job ID: f4d8a3dc-a06c-46d0-8ca3-b546001b347c


##  Retrieve Outputs

This cell list outputs under your S3 `out_dir` and optionally copy them into ADE. This loop gives you lightweight observability so you can proceed as soon as artifacts are ready.


In [13]:
s3_out = f"s3://maap-ops-workspace/{USER}/gee_exports/sat_emb_{YEAR}/"
print("Outputs under:", s3_out)
os.system(f"aws s3 ls {s3_out}")


ade_out = f"/projects/{USER}/gee_exports/sat_emb_{YEAR}/"
os.makedirs(ade_out, exist_ok=True)
os.system(f"aws s3 cp --recursive {s3_out} {ade_out}")
print("Copied to ADE:", ade_out)


Outputs under: s3://maap-ops-workspace/harshinigirish/gee_exports/sat_emb_2024/
Copied to ADE: /projects/harshinigirish/gee_exports/sat_emb_2024/


## Explore the Image Collection

Filter the collection to `YEAR`, then fetch `system:id` and `system:time_start`and builds a small table to sanity-check tile counts and dates. These quick checks confirm you pulled the intended temporal slice before heavier analysis.

In [12]:
AOI = ee.Geometry.Rectangle([-123.6, 36.8, -121.3, 38.4])

img = (ee.ImageCollection(ASSET_PATH)
       .filter(ee.Filter.calendarRange(YEAR, YEAR, 'year'))
       .filterBounds(AOI)
       .mosaic()
       .clip(AOI))

print("bands:", img.bandNames().size().getInfo())         # expect 64
print("first 5 bands:", img.bandNames().slice(0, 5).getInfo())


bands: 64
first 5 bands: ['A00', 'A01', 'A02', 'A03', 'A04']


 ## Sanity-Check Bands

This prints band count (expect **64**) and preview a few band names. A correct 64-band result validates that you’re working with the embeddings product, not a different imagery set.

In [11]:

col = (ee.ImageCollection(ASSET_PATH)
       .filter(ee.Filter.calendarRange(YEAR, YEAR, 'year')))

tiles = col.size().getInfo()
print("tiles:", tiles)

ids   = col.aggregate_array('system:id').getInfo()
times = col.aggregate_array('system:time_start').getInfo()

df = pd.DataFrame({'system_id': ids, 'date': pd.to_datetime(times, unit='ms')})
df.head()


tiles: 11073


,system_id,date
0,GOOGLE/SATELLITE_EMBEDDING/V1/ANNUAL/x006zcaic...,2024-01-01
1,GOOGLE/SATELLITE_EMBEDDING/V1/ANNUAL/x007p6pbl...,2024-01-01
2,GOOGLE/SATELLITE_EMBEDDING/V1/ANNUAL/x008q5bao...,2024-01-01
3,GOOGLE/SATELLITE_EMBEDDING/V1/ANNUAL/x008wqfez...,2024-01-01
4,GOOGLE/SATELLITE_EMBEDDING/V1/ANNUAL/x00hdzudd...,2024-01-01
